In [6]:
import pandas as pd

df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (71523, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0.0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0.0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0.0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0.0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0.0,1/15/2018 11:24,Standard Class


In [7]:
print("Missing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nLate Delivery Distribution:")
print(df['Late_delivery_risk'].value_counts())

print("\nLate Delivery Percentage:")
print(df['Late_delivery_risk'].value_counts(normalize=True) * 100)

Missing Values:
Product Description           71523
Order Zipcode                 62987
Customer Zipcode                  2
Customer Lname                    2
Product Status                    1
Product Name                      1
Product Price                     1
Shipping Mode                     1
shipping date (DateOrders)        1
Sales per customer                0
Customer Country                  0
Category Id                       0
Category Name                     0
Customer Fname                    0
Benefit per order                 0
dtype: int64

Late Delivery Distribution:
Late_delivery_risk
1    41905
0    29618
Name: count, dtype: int64

Late Delivery Percentage:
Late_delivery_risk
1    58.589545
0    41.410455
Name: proportion, dtype: float64


In [8]:
df = df.drop(columns=[
    'Product Description',
    'Order Zipcode',
    'Delivery Status',
    'Days for shipping (real)'
])

print("Dataset Shape:", df.shape)

Dataset Shape: (71523, 49)


In [9]:
features = [
    'Shipping Mode',
    'Days for shipment (scheduled)',
    'Order Region',
    'Order Item Quantity',
    'Order Item Total',
    'Category Name'
]

target = 'Late_delivery_risk'

data = df[features + [target]].copy()

print(data.shape)
print(data.isnull().sum())

(71523, 7)
Shipping Mode                    1
Days for shipment (scheduled)    0
Order Region                     0
Order Item Quantity              0
Order Item Total                 0
Category Name                    0
Late_delivery_risk               0
dtype: int64


In [11]:
print("Original dataset:", df.shape)
print("Selected dataset:", data.shape)

data = data.dropna()

print("After cleaning:", data.shape)
print(data['Late_delivery_risk'].value_counts())

Original dataset: (71523, 49)
Selected dataset: (71522, 7)
After cleaning: (71522, 7)
Late_delivery_risk
1    41905
0    29617
Name: count, dtype: int64


In [12]:
X = data.drop('Late_delivery_risk', axis=1)
y = data['Late_delivery_risk']

X = pd.get_dummies(X, dtype=int)

print("Features after encoding:", X.shape)
X.head()

Features after encoding: (71522, 79)


,Days for shipment (scheduled),Order Item Quantity,Order Item Total,Shipping Mode_First Class,Shipping Mode_Same Day,Shipping Mode_Second Class,Shipping Mode_Standard Class,Order Region_Canada,Order Region_Caribbean,Order Region_Central Africa,...,Category Name_Sporting Goods,Category Name_Strength Training,Category Name_Tennis & Racquet,Category Name_Toys,Category Name_Trade-In,Category Name_Video Games,Category Name_Water Sports,Category Name_Women's Apparel,Category Name_Women's Clothing,Category Name_Women's Golf Clubs
0,4,1,314.640015,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,4,1,311.359985,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,4,1,309.720001,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,4,1,304.809998,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,4,1,298.250000,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (57217, 79)
Testing: (14305, 79)


In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [15]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7140160782943027
ROC-AUC: 0.7636002215873483

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.84      0.71      5924
           1       0.85      0.63      0.72      8381

    accuracy                           0.71     14305
   macro avg       0.73      0.73      0.71     14305
weighted avg       0.75      0.71      0.71     14305


Confusion Matrix:
[[4962  962]
 [3129 5252]]


In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.3, 0.4, 0.5]:
    predictions = (y_prob >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", round(precision_score(y_test, predictions), 3))
    print("Recall:", round(recall_score(y_test, predictions), 3))
    print("F1 Score:", round(f1_score(y_test, predictions), 3))
    print()

Threshold: 0.3
Precision: 0.586
Recall: 0.997
F1 Score: 0.738

Threshold: 0.4
Precision: 0.751
Recall: 0.71
F1 Score: 0.73

Threshold: 0.5
Precision: 0.845
Recall: 0.627
F1 Score: 0.72



In [17]:
print("Total records:", len(data))

print("Unique orders:", df.loc[data.index, 'Order Id'].nunique())

print("Repeated order records:",
      df.loc[data.index, 'Order Id'].duplicated().sum())

Total records: 71522
Unique orders: 35792
Repeated order records: 35730


In [18]:
from sklearn.model_selection import GroupShuffleSplit

groups = df.loc[data.index, 'Order Id']

split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    split.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (57158, 79)
Testing: (14364, 79)


In [19]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [20]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix
)

y_prob = model.predict_proba(X_test)[:, 1]

# Use our candidate threshold
y_pred = (y_prob >= 0.4).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6970899470899471
ROC-AUC: 0.7705885074118466

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.65      0.64      5916
           1       0.75      0.73      0.74      8448

    accuracy                           0.70     14364
   macro avg       0.69      0.69      0.69     14364
weighted avg       0.70      0.70      0.70     14364


Confusion Matrix:
[[3864 2052]
 [2299 6149]]


In [21]:
import numpy as np

results = X_test.copy()

results['Actual_Delay'] = y_test
results['Delay_Probability'] = y_prob

results['Risk_Level'] = np.select(
    [
        results['Delay_Probability'] >= 0.70,
        results['Delay_Probability'] >= 0.40
    ],
    ['High Risk', 'Medium Risk'],
    default='Low Risk'
)

results[[
    'Actual_Delay',
    'Delay_Probability',
    'Risk_Level'
]].head(10)

,Actual_Delay,Delay_Probability,Risk_Level
5,0,0.375420,Low Risk
12,1,0.809971,High Risk
13,1,0.949439,High Risk
19,0,0.672370,Medium Risk
20,0,0.672370,Medium Risk
26,1,0.790268,High Risk
31,1,0.384498,Low Risk
35,1,0.976024,High Risk
50,1,0.779522,High Risk
56,1,0.782528,High Risk


In [22]:
results['Recommendation'] = 'Monitor shipment normally'

results.loc[
    results['Risk_Level'] == 'Medium Risk',
    'Recommendation'
] = 'Contact carrier and verify delivery schedule'

results.loc[
    results['Risk_Level'] == 'High Risk',
    'Recommendation'
] = 'Urgent: Contact carrier and review expedited shipping options'

results[[
    'Delay_Probability',
    'Risk_Level',
    'Recommendation'
]].head(10)

,Delay_Probability,Risk_Level,Recommendation
5,0.375420,Low Risk,Monitor shipment normally
12,0.809971,High Risk,Urgent: Contact carrier and review expedited s...
13,0.949439,High Risk,Urgent: Contact carrier and review expedited s...
19,0.672370,Medium Risk,Contact carrier and verify delivery schedule
20,0.672370,Medium Risk,Contact carrier and verify delivery schedule
26,0.790268,High Risk,Urgent: Contact carrier and review expedited s...
31,0.384498,Low Risk,Monitor shipment normally
35,0.976024,High Risk,Urgent: Contact carrier and review expedited s...
50,0.779522,High Risk,Urgent: Contact carrier and review expedited s...
56,0.782528,High Risk,Urgent: Contact carrier and review expedited s...


In [23]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
})

importance = importance.sort_values(
    'Importance',
    ascending=False
)

importance.head(15)

,Feature,Importance
0,Days for shipment (scheduled),0.284846
6,Shipping Mode_Standard Class,0.237928
3,Shipping Mode_First Class,0.218053
5,Shipping Mode_Second Class,0.086255
2,Order Item Total,0.083328
4,Shipping Mode_Same Day,0.031000
1,Order Item Quantity,0.010785
20,Order Region_South Asia,0.001930
29,Order Region_Western Europe,0.001772
10,Order Region_Central America,0.001603


In [24]:
results['Priority_Score'] = (
    results['Delay_Probability'] * 100
).round(2)

results['Priority_Rank'] = (
    results['Priority_Score']
    .rank(method='dense', ascending=False)
    .astype(int)
)

priority_shipments = results.sort_values(
    'Priority_Score',
    ascending=False
)

priority_shipments[[
    'Delay_Probability',
    'Priority_Score',
    'Priority_Rank',
    'Risk_Level',
    'Recommendation'
]].head(10)

,Delay_Probability,Priority_Score,Priority_Rank,Risk_Level,Recommendation
62822,0.989257,98.93,1,High Risk,Urgent: Contact carrier and review expedited s...
6016,0.988627,98.86,2,High Risk,Urgent: Contact carrier and review expedited s...
18908,0.988627,98.86,2,High Risk,Urgent: Contact carrier and review expedited s...
15745,0.987929,98.79,3,High Risk,Urgent: Contact carrier and review expedited s...
42412,0.987396,98.74,4,High Risk,Urgent: Contact carrier and review expedited s...
18949,0.987396,98.74,4,High Risk,Urgent: Contact carrier and review expedited s...
42405,0.987444,98.74,4,High Risk,Urgent: Contact carrier and review expedited s...
18922,0.987444,98.74,4,High Risk,Urgent: Contact carrier and review expedited s...
18927,0.987396,98.74,4,High Risk,Urgent: Contact carrier and review expedited s...
26690,0.987396,98.74,4,High Risk,Urgent: Contact carrier and review expedited s...


In [25]:
results['Shipping_Mode'] = df.loc[results.index, 'Shipping Mode']

results['Recommendation'] = 'Monitor shipment normally'

results.loc[
    results['Risk_Level'] == 'Medium Risk',
    'Recommendation'
] = 'Contact carrier and confirm shipping schedule'

results.loc[
    results['Risk_Level'] == 'High Risk',
    'Recommendation'
] = 'Urgent: Contact carrier and monitor shipment closely'

results.loc[
    (results['Risk_Level'] == 'High Risk') &
    (results['Shipping_Mode'] == 'Standard Class'),
    'Recommendation'
] = 'Review faster shipping options and contact carrier'

results.loc[
    (results['Risk_Level'] == 'High Risk') &
    (results['Shipping_Mode'] == 'First Class'),
    'Recommendation'
] = 'Escalate to carrier and verify dispatch priority'

results.loc[
    (results['Risk_Level'] == 'High Risk') &
    (results['Days for shipment (scheduled)'] <= 1),
    'Recommendation'
] = 'Urgent: Verify same-day dispatch and escalate delays'

results[[
    'Shipping_Mode',
    'Delay_Probability',
    'Risk_Level',
    'Recommendation'
]].head(10)

,Shipping_Mode,Delay_Probability,Risk_Level,Recommendation
5,Standard Class,0.375420,Low Risk,Monitor shipment normally
12,Second Class,0.809971,High Risk,Urgent: Contact carrier and monitor shipment c...
13,First Class,0.949439,High Risk,Urgent: Verify same-day dispatch and escalate ...
19,Same Day,0.672370,Medium Risk,Contact carrier and confirm shipping schedule
20,Same Day,0.672370,Medium Risk,Contact carrier and confirm shipping schedule
26,Second Class,0.790268,High Risk,Urgent: Contact carrier and monitor shipment c...
31,Standard Class,0.384498,Low Risk,Monitor shipment normally
35,First Class,0.976024,High Risk,Urgent: Verify same-day dispatch and escalate ...
50,Second Class,0.779522,High Risk,Urgent: Contact carrier and monitor shipment c...
56,Second Class,0.782528,High Risk,Urgent: Contact carrier and monitor shipment c...


In [26]:
results['Order_ID'] = df.loc[results.index, 'Order Id']

results['Order_Value'] = df.loc[results.index, 'Order Item Total']

final_report = results[[
    'Order_ID',
    'Shipping_Mode',
    'Order_Value',
    'Delay_Probability',
    'Priority_Score',
    'Priority_Rank',
    'Risk_Level',
    'Recommendation'
]].copy()

final_report = final_report.sort_values(
    'Priority_Score',
    ascending=False
)

final_report.head(10)

,Order_ID,Shipping_Mode,Order_Value,Delay_Probability,Priority_Score,Priority_Rank,Risk_Level,Recommendation
62822,23842,First Class,95.949997,0.989257,98.93,1,High Risk,Urgent: Verify same-day dispatch and escalate ...
6016,28692,First Class,299.980011,0.988627,98.86,2,High Risk,Urgent: Verify same-day dispatch and escalate ...
18908,20637,First Class,299.980011,0.988627,98.86,2,High Risk,Urgent: Verify same-day dispatch and escalate ...
15745,28692,First Class,120.919998,0.987929,98.79,3,High Risk,Urgent: Verify same-day dispatch and escalate ...
42412,22668,First Class,363.980011,0.987396,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
18949,21422,First Class,335.980011,0.987396,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
42405,27310,First Class,284.980011,0.987444,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
18922,21422,First Class,284.980011,0.987444,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
18927,20637,First Class,377.980011,0.987396,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
26690,23391,First Class,335.980011,0.987396,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...


In [27]:
order_report = final_report.groupby('Order_ID').agg(
    Shipping_Mode=('Shipping_Mode', 'first'),
    Order_Value=('Order_Value', 'sum'),
    Delay_Probability=('Delay_Probability', 'max')
).reset_index()

order_report['Priority_Score'] = (
    order_report['Delay_Probability'] * 100
).round(2)

order_report['Risk_Level'] = 'Low Risk'

order_report.loc[
    order_report['Delay_Probability'] >= 0.40,
    'Risk_Level'
] = 'Medium Risk'

order_report.loc[
    order_report['Delay_Probability'] >= 0.70,
    'Risk_Level'
] = 'High Risk'

order_report = order_report.sort_values(
    'Priority_Score',
    ascending=False
)

order_report.head(10)


,Order_ID,Shipping_Mode,Order_Value,Delay_Probability,Priority_Score,Risk_Level
2226,23842,First Class,547.759995,0.989257,98.93,High Risk
1942,20637,First Class,1177.910034,0.988627,98.86,High Risk
2619,28692,First Class,896.380013,0.988627,98.86,High Risk
2188,23391,First Class,694.760010,0.987396,98.74,High Risk
2313,25114,First Class,335.980011,0.987396,98.74,High Risk
2501,27310,First Class,543.860008,0.987444,98.74,High Risk
2046,21810,First Class,653.640015,0.987396,98.74,High Risk
2014,21422,First Class,1040.280029,0.987444,98.74,High Risk
2124,22668,First Class,891.920029,0.987396,98.74,High Risk
2312,25113,First Class,674.450012,0.987278,98.73,High Risk


In [28]:
recommendations = results.groupby('Order_ID')['Recommendation'].first()

order_report['Recommendation'] = (
    order_report['Order_ID'].map(recommendations)
)

order_report['Priority_Rank'] = (
    order_report['Delay_Probability']
    .rank(method='dense', ascending=False)
    .astype(int)
)

order_report[[
    'Order_ID',
    'Shipping_Mode',
    'Order_Value',
    'Priority_Score',
    'Priority_Rank',
    'Risk_Level',
    'Recommendation'
]].head(10)

,Order_ID,Shipping_Mode,Order_Value,Priority_Score,Priority_Rank,Risk_Level,Recommendation
2226,23842,First Class,547.759995,98.93,1,High Risk,Urgent: Verify same-day dispatch and escalate ...
1942,20637,First Class,1177.910034,98.86,2,High Risk,Urgent: Verify same-day dispatch and escalate ...
2619,28692,First Class,896.380013,98.86,2,High Risk,Urgent: Verify same-day dispatch and escalate ...
2188,23391,First Class,694.760010,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
2313,25114,First Class,335.980011,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
2501,27310,First Class,543.860008,98.74,3,High Risk,Urgent: Verify same-day dispatch and escalate ...
2046,21810,First Class,653.640015,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
2014,21422,First Class,1040.280029,98.74,3,High Risk,Urgent: Verify same-day dispatch and escalate ...
2124,22668,First Class,891.920029,98.74,4,High Risk,Urgent: Verify same-day dispatch and escalate ...
2312,25113,First Class,674.450012,98.73,5,High Risk,Urgent: Verify same-day dispatch and escalate ...


In [29]:
reliability = pd.DataFrame({
    'Actual_Delay': y_test,
    'Predicted_Probability': y_prob
})

reliability['Risk_Group'] = pd.cut(
    reliability['Predicted_Probability'],
    bins=[0, 0.4, 0.7, 1],
    labels=['Low Risk', 'Medium Risk', 'High Risk'],
    include_lowest=True
)

reliability.groupby(
    'Risk_Group',
    observed=True
).agg(
    Shipments=('Actual_Delay', 'count'),
    Actual_Delay_Rate=('Actual_Delay', 'mean'),
    Average_Predicted_Risk=('Predicted_Probability', 'mean')
).round(3)

,Shipments,Actual_Delay_Rate,Average_Predicted_Risk
Risk_Group,,,
Low Risk,6163,0.373,0.374
Medium Risk,2385,0.428,0.456
High Risk,5816,0.882,0.867


In [30]:
import joblib

joblib.dump(model, 'cargorisk_model.pkl')

joblib.dump(
    X_train.columns.tolist(),
    'model_features.pkl'
)

print("Model and features saved successfully!")

Model and features saved successfully!
